# Trend-Equity — Prompt & Feature Tester

This notebook lets you call the **same Gemini prompts** used by Trend-Equity, directly — no server, no browser required.

**Run order:**
1. `[Setup]` — install + configure API key
2. `[Core]` — system prompt + helpers (run once per session)
3. Any feature section in any order

---
**Features covered:**
| Section | Feature |
|---|---|
| A | Daily Ideas (3 batches → 35 ideas) |
| B | Trend Radar |
| C | Futurecasting |
| D | Market Alerts |
| E | Explain |
| F | VC Vetting |
| G | Validation Toolkit |
| H | Build-Me Starter Pack |
| I | Action Plan |
| J | Analyze Your Own Idea |

---
## Setup
Run these two cells once at the start of every session.

In [ ]:
# Install the Google Gen AI SDK (same package family as @google/genai used by the app)
!pip install -q google-genai

In [ ]:
import google.genai as genai
from google.genai import types
import json
from datetime import datetime

# ---------------------------------------------------------------------------
# API Key — two options:
#   Option 1 (Colab):  Add a secret named GEMINI_API_KEY in the key icon (🔑)
#                      on the left sidebar, then uncomment the line below.
#   Option 2 (manual): Paste your key directly as a string.
# ---------------------------------------------------------------------------

# from google.colab import userdata
# GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

GEMINI_API_KEY = ""   # <-- paste your key here if not using Colab secrets

MODEL = "gemini-1.5-flash"  # change to "gemini-2.5-flash" if you have access

if not GEMINI_API_KEY:
    raise ValueError("Set GEMINI_API_KEY above before continuing.")

client = genai.Client(api_key=GEMINI_API_KEY)
print(f"Client ready. Model: {MODEL}")

---
## Core — System Prompt & Helpers
This is a direct copy of the production system prompt and schema definitions from `api/_lib/ai-provider.ts`.

In [ ]:
# Copied verbatim from api/_lib/ai-provider.ts → DEFAULT_SYSTEM_PROMPT
DEFAULT_SYSTEM_PROMPT = """You are Trend-Equity's principal venture scout — a seasoned analyst with deep expertise in early-stage VC, product strategy, and emerging markets. Your mission: surface business ideas that serious founders and investors would act on today.

QUALITY STANDARDS — every idea must pass all three tests:
1. SIGNAL-GROUNDED: Cite a specific, verifiable market event, data point, or regulatory shift from 2025–2026 in trendSources. Generic trends ("AI is growing") are not acceptable.
2. NON-OBVIOUS: Target second or third-order opportunities created by the trend — not the obvious direct play. If everyone sees the trend, find the problem it creates that nobody is solving yet.
3. STRUCTURAL EDGE: The unfairAdvantage field must describe a real, defensible moat — proprietary data, regulatory position, distribution lock-in, or network effects. "Better UX" and "first mover advantage" are not moats.

DIVERSITY RULES:
- Span at least 7 distinct verticals per batch (AI/ML, FinTech, HealthTech, EdTech, CleanTech, Consumer/D2C, B2B SaaS, Marketplace, PropTech, AgriTech, LegalTech, GovTech, Creator Economy, etc.)
- No more than 3 ideas from any single sector
- Mix effort levels: at least 8 solo-buildable ideas (<6 weeks), at least 8 small-team ideas, the rest for well-funded teams
- At least 20% of ideas should address markets outside the US or EU

PROHIBITED PATTERNS — skip or fundamentally reimagine any idea matching these templates:
✗ Generic "AI copilot/assistant for [profession]" without proprietary training data or workflow lock-in
✗ Another productivity, note-taking, task management, or team collaboration tool
✗ Crypto/Web3/NFT without a concrete non-speculative revenue model validated by real users
✗ "Uber/Airbnb for X" without a clear structural reason why aggregation has not worked in that vertical
✗ Carbon/ESG reporting SaaS without a specific compliance mandate creating immediate spend
✗ General-purpose AI chatbot integration for any industry without a proprietary data advantage

QUALITY BAR: Before finalising each idea, ask — "Would a seed-stage partner at a top-10 VC firm pause and want to learn more?" If the idea sounds like something that was well-funded 3–4 years ago, replace it with something fresher.

OUTPUT FORMAT: Respond with valid JSON matching the provided schema exactly. No markdown, no commentary outside the JSON."""

# Extended system prompt used only for Analyze-Your-Own-Idea (Section J)
ANALYZE_IDEA_SYSTEM_PROMPT = DEFAULT_SYSTEM_PROMPT + (
    "\n\nThe user has described their own business idea. Your job is to rigorously evaluate it — "
    "not to encourage it. Apply your full VC-grade critical framework. Be honest about market "
    "saturation, competitive dynamics, and structural risks."
)

print("System prompts loaded.")

In [ ]:
def generate_with_ai(prompt, schema=None, system_instruction=None):
    """
    Central generation helper — mirrors generateWithAI() in api/_lib/ai-provider.ts.
    Returns parsed dict when schema is provided, plain string otherwise.
    """
    sys_prompt = system_instruction or DEFAULT_SYSTEM_PROMPT

    # Append the strict JSON reminder the app uses when a schema is present
    final_prompt = (
        prompt
        + "\n\nIMPORTANT: Return ONLY a raw JSON object matching the requested schema. "
          "No markdown code fences, no preamble, no conversational text. Start with { and end with }."
        if schema else prompt
    )

    config = types.GenerateContentConfig(
        system_instruction=sys_prompt,
        response_mime_type="application/json" if schema else "text/plain",
        response_schema=schema,
    )

    response = client.models.generate_content(
        model=MODEL,
        contents=final_prompt,
        config=config,
    )

    text = response.text
    if schema:
        # Strip markdown fences if any (mirrors cleanJSON() in the app)
        import re
        text = text.strip()
        fence_match = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", text, re.IGNORECASE)
        if fence_match:
            text = fence_match.group(1)
        if not text.startswith(("{", "[")):
            start = text.find("{")
            end = text.rfind("}")
            if start != -1 and end > start:
                text = text[start:end + 1]
        return json.loads(text)
    return text


def pretty(data):
    """Pretty-print a dict/list."""
    print(json.dumps(data, indent=2))


def today():
    return datetime.utcnow().strftime("%Y-%m-%d")


print("Helpers ready.")

In [ ]:
# ---------------------------------------------------------
# JSON Schemas — ported from api/_lib/ai-provider.ts
# ---------------------------------------------------------

IDEA_SCHEMA = {
    "type": "object",
    "properties": {
        "id":                    {"type": "string"},
        "headline":              {"type": "string"},
        "pitch":                 {"type": "string"},
        "vcJustification":       {"type": "string"},
        "categoryTags":          {"type": "array", "items": {"type": "string"}},
        "costEffort":            {"type": "string"},
        "revenuePotentialScore": {"type": "number"},
        "revenueSkeleton":       {"type": "string"},
        "unfairAdvantage":       {"type": "string"},
        "potentialExit":         {"type": "string"},
        "trendSources":          {"type": "array", "items": {"type": "string"}},
        "saturationLabel":       {"type": "string"},
        "heatBadge":             {"type": "string"},
        "nextSteps":             {"type": "array", "items": {"type": "string"}},
        "competitorLandscape":   {"type": "string"},
        "regulatoryFlags":       {"type": "string"},
        "marketSize":            {"type": "string"},
    },
    "required": [
        "headline", "pitch", "vcJustification", "categoryTags", "costEffort",
        "revenuePotentialScore", "revenueSkeleton", "unfairAdvantage", "potentialExit",
        "trendSources", "saturationLabel", "heatBadge", "nextSteps",
        "marketSize", "competitorLandscape", "regulatoryFlags",
    ],
}

DAILY_SCHEMA = {
    "type": "object",
    "properties": {
        "intro":      {"type": "string"},
        "ideas":      {"type": "array", "items": IDEA_SCHEMA},
        "disclaimer": {"type": "string"},
    },
    "required": ["intro", "ideas", "disclaimer"],
}

RADAR_SCHEMA = {
    "type": "object",
    "properties": {
        "week": {"type": "string"},
        "marketShift": {
            "type": "object",
            "properties": {
                "title":       {"type": "string"},
                "description": {"type": "string"},
            },
            "required": ["title", "description"],
        },
        "topTrends": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "title":       {"type": "string"},
                    "description": {"type": "string"},
                    "impact":      {"type": "string"},
                    "sector":      {"type": "string"},
                },
                "required": ["title", "description", "impact", "sector"],
            },
        },
        "opportunityAreas": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["week", "marketShift", "topTrends", "opportunityAreas"],
}

FUTURECASTING_SCHEMA = {
    "type": "object",
    "properties": {
        "horizon": {"type": "string"},
        "predictions": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "title":       {"type": "string"},
                    "probability": {"type": "number"},
                    "rationale":   {"type": "string"},
                    "winners":     {"type": "array", "items": {"type": "string"}},
                    "losers":      {"type": "array", "items": {"type": "string"}},
                },
                "required": ["title", "probability", "rationale", "winners", "losers"],
            },
        },
        "paradigmShifts": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "title":     {"type": "string"},
                    "rationale": {"type": "string"},
                    "impact":    {"type": "string"},
                },
                "required": ["title", "rationale", "impact"],
            },
        },
    },
    "required": ["horizon", "predictions", "paradigmShifts"],
}

ALERTS_SCHEMA = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "title":   {"type": "string"},
            "message": {"type": "string"},
            "type":    {"type": "string", "enum": ["info", "success", "warning", "error"]},
        },
        "required": ["title", "message", "type"],
    },
}

VETTING_SCHEMA = {
    "type": "object",
    "properties": {
        "score":             {"type": "number"},
        "verdict":           {"type": "string", "enum": ["High Conviction", "Moderate", "Pass"]},
        "strengths":         {"type": "array", "items": {"type": "string"}},
        "weaknesses":        {"type": "array", "items": {"type": "string"}},
        "riskMitigation":    {"type": "array", "items": {"type": "string"}},
        "pivotSuggestions":  {"type": "array", "items": {"type": "string"}},
        "comparableExits":   {"type": "array", "items": {"type": "string"}},
    },
    "required": ["score", "verdict", "strengths", "weaknesses",
                 "riskMitigation", "pivotSuggestions", "comparableExits"],
}

VALIDATION_SCHEMA = {
    "type": "object",
    "properties": {
        "landingPage": {
            "type": "object",
            "properties": {
                "hero":       {"type": "string"},
                "subHero":    {"type": "string"},
                "valueProps": {"type": "array", "items": {"type": "string"}},
            },
            "required": ["hero", "subHero", "valueProps"],
        },
        "interviewScript":  {"type": "array", "items": {"type": "string"}},
        "smokeTest":        {"type": "string"},
        "successMetrics":   {"type": "array", "items": {"type": "string"}},
    },
    "required": ["landingPage", "interviewScript", "smokeTest", "successMetrics"],
}

BUILD_ME_SCHEMA = {
    "type": "object",
    "properties": {
        "promptPack": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "title":  {"type": "string"},
                    "prompt": {"type": "string"},
                },
                "required": ["title", "prompt"],
            },
        },
        "repoStructure": {"type": "string"},
        "first24Hours":  {"type": "array", "items": {"type": "string"}},
    },
    "required": ["promptPack", "repoStructure", "first24Hours"],
}

ACTION_PLAN_SCHEMA = {
    "type": "object",
    "properties": {
        "roadmap": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "id":        {"type": "string"},
                    "step":      {"type": "string"},
                    "details":   {"type": "string"},
                    "milestone": {"type": "string"},
                },
                "required": ["id", "step", "details", "milestone"],
            },
        },
        "tools":    {"type": "array", "items": {"type": "string"}},
        "risks":    {"type": "array", "items": {"type": "string"}},
        "timeline": {"type": "string"},
    },
    "required": ["roadmap", "tools", "risks", "timeline"],
}

print("Schemas loaded.")

---
## A — Daily Ideas
Replicates the 3-batch parallel generation from `api/generate/daily.ts`.
Each batch targets a different category cluster. In production these run concurrently;
here they run sequentially (same prompts, same schema).

**Optional:** paste custom market signals into `CUSTOM_SIGNALS` to ground the ideas in real data.

In [ ]:
# Optional: add your own market signals (leave empty to skip)
CUSTOM_SIGNALS = """
# Example — replace or leave blank
# - OpenAI released o3 API in May 2026, reducing inference cost by 40%
# - EU AI Act enforcement begins June 2026 for high-risk systems
# - India's UPI processed $2.4T in transactions in Q1 2026
""".strip()

DEDUPE_BLOCK = ""  # paste recent headlines here to avoid duplicates
COUNTRY_CLAUSE = ""  # e.g. "Additionally, include 5 ideas for the Indian market."

QUALITY_BLOCK = """

QUALITY ENFORCEMENT:
- No more than 3 ideas per sector
- At least 7 distinct verticals
- Mix effort levels: solo-buildable, small-team, and well-funded
- At least 20% of ideas outside US/EU markets
- Avoid: generic AI assistants without proprietary data, copycat marketplaces,
  basic CRUD tools, Web3 without real revenue model
""".strip()

print("Daily config ready.")

In [ ]:
# Batch 1 — Digital / SaaS / AI-SaaS  (12 ideas)
signal_prefix = f"{CUSTOM_SIGNALS}\n\n" if CUSTOM_SIGNALS else ""
dedupe = f"\n\nDO NOT REPEAT RECENT IDEAS:\n{DEDUPE_BLOCK}" if DEDUPE_BLOCK else ""
country = f"\n\n{COUNTRY_CLAUSE}" if COUNTRY_CLAUSE else ""

prompt_batch1 = (
    f"{signal_prefix}"
    f"{dedupe}{country}\n{QUALITY_BLOCK}\n\n"
    f"{'Using the live market signals above as your PRIMARY source, generate' if CUSTOM_SIGNALS else 'Generate'} "
    f"exactly 12 high-conviction business ideas for {today()}. "
    f"Focus specifically on the following categories: 'Digital / SaaS / AI-SaaS'."
)

print("Calling Gemini — Batch 1 (Digital/SaaS/AI-SaaS)...")
batch1 = generate_with_ai(prompt_batch1, DAILY_SCHEMA)
print(f"Got {len(batch1.get('ideas', []))} ideas")
pretty(batch1)

In [ ]:
# Batch 2 — Service / Local / On-Demand + Wildcard  (12 ideas)
prompt_batch2 = (
    f"{signal_prefix}"
    f"{dedupe}{country}\n{QUALITY_BLOCK}\n\n"
    f"{'Using the live market signals above as your PRIMARY source, generate' if CUSTOM_SIGNALS else 'Generate'} "
    f"exactly 12 high-conviction business ideas for {today()}. "
    f"Focus specifically on the following categories: 'Service / Local / On-Demand', 'Wildcard (creative/misc)'."
)

print("Calling Gemini — Batch 2 (Service/Local/Wildcard)...")
batch2 = generate_with_ai(prompt_batch2, DAILY_SCHEMA)
print(f"Got {len(batch2.get('ideas', []))} ideas")
pretty(batch2)

In [ ]:
# Batch 3 — Physical / Sustainable / Hardware + Deep-Tech / Moonshot  (11 ideas)
prompt_batch3 = (
    f"{signal_prefix}"
    f"{dedupe}{country}\n{QUALITY_BLOCK}\n\n"
    f"{'Using the live market signals above as your PRIMARY source, generate' if CUSTOM_SIGNALS else 'Generate'} "
    f"exactly 11 high-conviction business ideas for {today()}. "
    f"Focus specifically on the following categories: 'Physical / Sustainable / Hardware', 'Deep-Tech / Moonshot'."
)

print("Calling Gemini — Batch 3 (Physical/DeepTech)...")
batch3 = generate_with_ai(prompt_batch3, DAILY_SCHEMA)
print(f"Got {len(batch3.get('ideas', []))} ideas")
pretty(batch3)

In [ ]:
# Merge all 3 batches — mirrors what the server does after Promise.all()
all_ideas = (
    batch1.get("ideas", []) +
    batch2.get("ideas", []) +
    batch3.get("ideas", [])
)

daily_feed = {
    "date": today(),
    "intro": batch1.get("intro") or batch2.get("intro") or "Today's high-signal ideas.",
    "ideas": all_ideas,
    "disclaimer": batch1.get("disclaimer") or "All ideas are AI-generated from real market signals.",
    "generatedAt": datetime.utcnow().isoformat(),
}

print(f"Total ideas generated: {len(all_ideas)}")

# Quick summary table
print(f"\n{'#':<4} {'Headline':<60} {'Score':<6} {'Heat'}")
print("-" * 90)
for i, idea in enumerate(all_ideas, 1):
    print(f"{i:<4} {idea.get('headline', '')[:58]:<60} {idea.get('revenuePotentialScore', '-'):<6} {idea.get('heatBadge', '')}")

---
## B — Trend Radar
Mirrors `api/generate/radar.ts`. Paste optional signals below.

In [ ]:
RADAR_SIGNALS = ""  # optional: paste signal context here

radar_instruction = (
    f"Perform an intensive VC-grade market analysis for the week of {today()}.\n"
    "1. Identify the single most significant MARKET SHIFT.\n"
    "2. Detail 5 TOP TRENDS with descriptions, sector focus, and specific market impact.\n"
    "3. List 5 OPPORTUNITY AREAS for builders and founders to target.\n"
    "Focus on real, high-signal data from the last 7 days."
)

radar_prompt = f"{RADAR_SIGNALS}\n\n{radar_instruction}" if RADAR_SIGNALS else radar_instruction

print("Calling Gemini — Trend Radar...")
radar = generate_with_ai(radar_prompt, RADAR_SCHEMA)
pretty(radar)

---
## C — Futurecasting
Mirrors `api/generate/futurecasting.ts`. Valid horizons: `2027`, `2030`, `2035`.

In [ ]:
HORIZON = "2030"  # change to "2027" or "2035"

assert HORIZON in ("2027", "2030", "2035"), "Horizon must be 2027, 2030, or 2035"

print(f"Calling Gemini — Futurecasting to {HORIZON}...")
futurecasting = generate_with_ai(
    f"Perform a deep-future technology and market simulation up to the year {HORIZON}. "
    "Provide 5 high-probability predictions with rationale, winners, and losers, "
    "plus 3 paradigm shifts.",
    FUTURECASTING_SCHEMA,
)
pretty(futurecasting)

---
## D — Market Alerts
Mirrors `api/generate/alerts.ts`.

In [ ]:
print("Calling Gemini — Market Alerts...")
alerts = generate_with_ai(
    "Generate 3-5 high-signal Market Trend Alerts.",
    ALERTS_SCHEMA,
)
pretty(alerts)

---
## E — Explain
Mirrors `api/generate/explain.ts`. Given an idea headline and a section name,
returns a plain-text explanation.

Typical section values: `Business Model`, `Target Market`, `Revenue Skeleton`,
`Unfair Advantage`, `Potential Exit`, `Regulatory Flags`.

In [ ]:
EXPLAIN_HEADLINE = "AI-Powered Regulatory Change Monitor for Mid-Market CFOs"
EXPLAIN_SECTION  = "Business Model"
EXPLAIN_CONTEXT  = ""  # optional extra context

explain_prompt = (
    f'Explain the "{EXPLAIN_SECTION}" aspect for this startup idea: {EXPLAIN_HEADLINE}'
    + (f". Additional context: {EXPLAIN_CONTEXT}" if EXPLAIN_CONTEXT else "")
)

print(f"Calling Gemini — Explain '{EXPLAIN_SECTION}'...")
explanation = generate_with_ai(explain_prompt)  # no schema → plain text
print(explanation)

---
## F — VC Vetting
Mirrors `api/generate/vetting.ts`. Score 1–100 + verdict + strengths/weaknesses.

In [ ]:
VETTING_IDEA = {
    "headline": "AI-Powered Regulatory Change Monitor for Mid-Market CFOs",
    "pitch":    "Real-time alerts on tax code and compliance rule changes, "
                "mapped automatically to a company's financial exposure.",
    "vcJustification": "Mid-market CFOs spend 30% of their time on reactive compliance research. "
                       "No incumbent solution auto-maps regulatory changes to balance sheet impact.",
}

vetting_prompt = (
    f"Perform expert VC-grade vetting for this startup idea: {VETTING_IDEA['headline']}\n\n"
    f"PITCH: {VETTING_IDEA['pitch']}\n"
    f"JUSTIFICATION: {VETTING_IDEA['vcJustification']}\n\n"
    "TASK:\n"
    "1. Score 1-100 and provide a verdict.\n"
    "2. List 3-5 core strengths.\n"
    "3. List 3-5 critical weaknesses/risks.\n"
    "4. For EACH weakness, provide a specific risk mitigation strategy.\n"
    "5. Provide 3 pivot suggestions if needed.\n"
    "6. List 3 comparable startup exits.\n\n"
    "IMPORTANT: Return riskMitigation as an array of strings corresponding to the weaknesses."
)

print("Calling Gemini — VC Vetting...")
vetting = generate_with_ai(vetting_prompt, VETTING_SCHEMA)
print(f"Score: {vetting.get('score')}/100 — {vetting.get('verdict')}")
pretty(vetting)

---
## G — Validation Toolkit
Mirrors `api/generate/validation.ts`. Generates landing page copy, interview script, smoke test, and success metrics.

In [ ]:
VALIDATION_IDEA = {
    "headline": "AI-Powered Regulatory Change Monitor for Mid-Market CFOs",
    "pitch":    "Real-time alerts on tax code and compliance rule changes, "
                "mapped automatically to a company's financial exposure.",
    "vcJustification": "Mid-market CFOs spend 30% of their time on reactive compliance research.",
}

validation_prompt = f"""
You are a specialized GTM (Go-To-Market) strategist. Create a professional validation toolkit for the following business idea:

HEADLINE: {VALIDATION_IDEA['headline']}
PITCH: {VALIDATION_IDEA['pitch']}
CONTEXT: {VALIDATION_IDEA['vcJustification']}

TASK:
Generate a validation strategy to prove market demand before building.
Include:
1. High-conversion landing page copy (hero, subhero, and 3 specific value props).
2. A 5-question problem-interview script for potential customers.
3. A specific 'smoke test' strategy (e.g. ad campaign, waitlist, pre-order).
4. 3 success metrics to track.

IMPORTANT: You MUST use camelCase for all keys (e.g., landingPage, interviewScript, smokeTest, successMetrics). Return valid JSON matching the schema.
""".strip()

print("Calling Gemini — Validation Toolkit...")
validation = generate_with_ai(validation_prompt, VALIDATION_SCHEMA)
pretty(validation)

---
## H — Build-Me Starter Pack
Mirrors `api/generate/build-me.ts`. Returns 3 AI prompts, a repo structure, and a 24-hour task list.

In [ ]:
BUILD_ME_IDEA = {
    "headline": "AI-Powered Regulatory Change Monitor for Mid-Market CFOs",
    "pitch":    "Real-time alerts on tax code and compliance rule changes, "
                "mapped automatically to a company's financial exposure.",
}

build_me_prompt = (
    f"Act as a Senior Technical Lead. Create a high-value \"Build-with-Me\" starter pack "
    f"for this startup idea: {BUILD_ME_IDEA['headline']}\n\n"
    f"PITCH: {BUILD_ME_IDEA['pitch']}\n\n"
    "TASK:\n"
    "1. PROMPT PACK: Provide 3 high-leverage AI prompts (System Design, Data Schema, and Core Feature Logic) "
    "that a developer can copy-paste into an LLM to start implementation.\n"
    "2. REPO STRUCTURE: Provide a clean, text-based visual tree of the recommended file structure "
    "(e.g., using ├── and └── characters). Do not return a JSON array.\n"
    "3. FIRST 24 HOURS: List 5-7 technical tasks to get a functional MVP live within one day.\n\n"
    "IMPORTANT:\n"
    "- Return exactly 3 prompts in the promptPack array.\n"
    "- repoStructure must be a single string formatted as a visual file tree."
)

print("Calling Gemini — Build-Me Starter Pack...")
build_me = generate_with_ai(build_me_prompt, BUILD_ME_SCHEMA)

print("\n=== Repo Structure ===")
print(build_me.get("repoStructure", ""))
print("\n=== First 24 Hours ===")
for i, task in enumerate(build_me.get("first24Hours", []), 1):
    print(f"  {i}. {task}")
print("\n=== Prompt Pack ===")
for p in build_me.get("promptPack", []):
    print(f"\n--- {p.get('title')} ---")
    print(p.get("prompt"))

---
## I — Action Plan
Mirrors `api/generate/action-plan.ts`. Generates a 4-phase roadmap (30-60-90 days + scaling).

In [ ]:
ACTION_PLAN_HEADLINE = "AI-Powered Regulatory Change Monitor for Mid-Market CFOs"

action_plan_prompt = f"""
You are Trend-Equity's principal product strategist. Generate a high-conviction implementation roadmap for the startup idea: "{ACTION_PLAN_HEADLINE}".

TASK:
Create a detailed 4-phase roadmap (30-60-90 day + scaling) and identifying necessary tools and risks.

REQUIRED JSON STRUCTURE:
- roadmap: An array of objects, each with:
  - id: string (e.g. "step-1")
  - step: string (Phase name or main task)
  - details: string (2-3 sentences of specific implementation advice)
  - milestone: string (A clear KPI or output for this step)
- tools: An array of 5-7 specific technical tools or platforms (e.g. "Vercel", "Supabase", "Stripe").
- risks: An array of 3-4 specific business or technical risks.
- timeline: A short string summarizing the total time to MVP.

IMPORTANT: You MUST return a valid JSON object matching the schema. No conversational text.
""".strip()

print("Calling Gemini — Action Plan...")
action_plan = generate_with_ai(action_plan_prompt, ACTION_PLAN_SCHEMA)
print(f"Timeline to MVP: {action_plan.get('timeline')}")
print(f"Tools: {', '.join(action_plan.get('tools', []))}")
print("\nRoadmap:")
for step in action_plan.get("roadmap", []):
    print(f"  [{step.get('id')}] {step.get('step')} → {step.get('milestone')}")
print("\nFull response:")
pretty(action_plan)

---
## J — Analyze Your Own Idea
Mirrors `api/generate/analyze-idea.ts`. Paste your own idea below.
Uses the stricter `ANALYZE_IDEA_SYSTEM_PROMPT` (critical VC evaluation mode).

In [ ]:
MY_IDEA = """
A platform that monitors government procurement portals across Southeast Asia
and uses AI to match SMEs with relevant tender opportunities, then auto-drafts
the initial bid response. Target: SMEs in SG, MY, TH that lack dedicated bid teams.
""".strip()

assert len(MY_IDEA) >= 20, "Describe your idea in at least 20 characters."

analyze_prompt = f"""Analyze the following business idea and produce a complete VC-grade evaluation.

The user's idea:
\"{MY_IDEA}\"

REQUIREMENTS:
- Treat this as a real early-stage startup pitch being evaluated for investment
- Cite a specific, verifiable 2025–2026 market signal or data point in trendSources that is directly relevant to this idea (positive OR negative)
- Identify the non-obvious second-order opportunity within the idea's space — what problem does this market create that is currently undersolved?
- The unfairAdvantage MUST describe a real structural moat THIS specific idea could realistically build (proprietary data, regulatory position, distribution lock-in, or network effects)
- Be honest about saturationLabel — if the space is crowded, say so clearly
- heatBadge must reflect the CURRENT market moment for this specific category
- nextSteps must be concretely tailored to this specific idea, not generic startup advice
- revenuePotentialScore must be a genuine assessment (1–10), not inflated
- competitorLandscape: identify 2-3 existing players or direct categories this idea competes with
- regulatoryFlags: Identify any legal, compliance, or regulatory hurdles this idea might face
- marketSize: Provide a specific estimate (e.g. $XB by 2030) or a description of the TAM based on current trends
- If the idea has fundamental problems, surface them honestly."""

print("Calling Gemini — Analyzing your idea...")
my_idea_result = generate_with_ai(analyze_prompt, IDEA_SCHEMA, ANALYZE_IDEA_SYSTEM_PROMPT)

print(f"Headline:       {my_idea_result.get('headline')}")
print(f"Revenue Score:  {my_idea_result.get('revenuePotentialScore')}/10")
print(f"Heat Badge:     {my_idea_result.get('heatBadge')}")
print(f"Saturation:     {my_idea_result.get('saturationLabel')}")
print(f"Market Size:    {my_idea_result.get('marketSize')}")
print("\nFull evaluation:")
pretty(my_idea_result)